Checklist

 - Add GNN layer (currently 2, need 3)
 - Reduce dimension of GNN layer
 - Add relevant features in featurizer
 - Check dimensions at each layer
 - Hyperparameter tuning

In [15]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from rdkit import Chem
from rdkit import RDLogger
from rdkit.Chem.Draw import IPythonConsole
from rdkit.Chem.Draw import MolsToGridImage

warnings.filterwarnings("ignore")
RDLogger.DisableLog("rdApp.*")

np.random.seed(42)

In [16]:
import torch
from torch.nn import Linear
import torch.nn.functional as F 
from torch_geometric.nn import GCNConv, TopKPooling
from torch_geometric.nn import global_mean_pool as gap, global_max_pool as gmp
embedding_size = 64

In [ ]:
import numpy as np
from rdkit import Chem
from rdkit.Chem.rdchem import BondType

class Featurizer:
    def __init__(self, allowable_sets, direct_features=None):
        """
        Base featurizer class. 
        - `allowable_sets`: Dict of categorical features (one-hot encoded).
        - `direct_features`: Dict of numerical features (used as direct values).
        """
        self.dim = 0
        self.features_mapping = {}
        self.direct_features = direct_features if direct_features else {}

        # One-hot encoded categorical features
        for k, s in allowable_sets.items():
            s = sorted(list(s))
            self.features_mapping[k] = dict(zip(s, range(self.dim, len(s) + self.dim)))
            self.dim += len(s)

        # Direct numerical features (not one-hot)
        for k in self.direct_features.keys():
            self.features_mapping[k] = self.dim
            self.dim += 1

    def encode(self, inputs):
        output = np.zeros((self.dim,))

        # One-hot encoded features
        for name_feature, feature_mapping in self.features_mapping.items():
            if name_feature in self.direct_features:  # Skip direct features in one-hot encoding
                continue
            feature = getattr(self, name_feature)(inputs)
            if feature not in feature_mapping:
                continue
            output[feature_mapping[feature]] = 1.0

        # Direct numerical features
        for name_feature, index in self.direct_features.items():
            feature = getattr(self, name_feature)(inputs)
            output[index] = feature  # Assign directly

        return output


class AtomFeaturizer(Featurizer):
    def __init__(self, allowable_sets, direct_features):
        super().__init__(allowable_sets, direct_features)

    def symbol(self, atom):
        return atom.GetSymbol()

    def n_hydrogens(self, atom):
        return atom.GetTotalNumHs() if atom.HasProp("_TotalNumHs") else 0

    def hybridization(self, atom):
        return atom.GetHybridization().name.lower()

    def is_aromatic(self, atom):
        return atom.GetIsAromatic()

    def n_valence(self, atom):
        return atom.GetTotalValence()

    def formal_charge(self, atom):
        return atom.GetFormalCharge()  # Direct feature, not one-hot

    def atomic_number(self, atom):
        return atom.GetAtomicNum()  # Direct feature


class BondFeaturizer(Featurizer):
    def __init__(self, allowable_sets, direct_features):
        super().__init__(allowable_sets, direct_features)

    def bond_type(self, bond):
        return bond.GetBondType().name.lower()

    def conjugated(self, bond):
        return bond.GetIsConjugated()

    def bond_order(self, bond):
        """Returns a numeric bond order instead of a categorical one-hot encoding."""
        bond_orders = {
            BondType.SINGLE: 1.0,
            BondType.DOUBLE: 2.0,
            BondType.TRIPLE: 3.0,
            BondType.AROMATIC: 1.5,
        }
        return bond_orders.get(bond.GetBondType(), 0.0)


# Instantiate with updated feature sets
atom_featurizer = AtomFeaturizer(
    allowable_sets={
        "symbol": {"B", "Br", "C", "Ca", "Cl", "F","Ga", "H", "I", "N", "Na", "O", "P", "S","Sb","Se", "Mo", "Nb"},
        "n_hydrogens": {0, 1, 2, 3, 4},
        "hybridization": {"s", "sp", "sp2", "sp3"},
    },
    direct_features={
        "formal_charge": None,  # This will be assigned a direct index in `Featurizer`
        "atomic_number": None,
        # "n_valence": None,  # Direct numeric feature
        "is_aromatic": None,  # Direct boolean feature
        "n_hydrogens": None,  # Direct numeric feature
    }
)

bond_featurizer = BondFeaturizer(
    allowable_sets={
        "bond_type": {"single", "double", "triple", "aromatic"},
        "conjugated": {True, False},
    },
    direct_features={
        "bond_order": None,  # Direct numeric feature
    }
)

# Get the new feature dimensions
node_dim = atom_featurizer.dim
edge_dim = bond_featurizer.dim

print(f"Node feature dimension: {node_dim}")
print(f"Edge feature dimension: {edge_dim}")


Node feature dimension: 31
Edge feature dimension: 7


In [18]:
import torch
import numpy as np
from rdkit import Chem
from torch_geometric.data import Data

# Ensure you define atom_featurizer and bond_featurizer before using them

def molecule_from_smiles(smiles):
    """Convert SMILES to RDKit molecule with error handling."""
    molecule = Chem.MolFromSmiles(smiles, sanitize=False)
    flag = Chem.SanitizeMol(molecule, catchErrors=True)
    if flag != Chem.SanitizeFlags.SANITIZE_NONE:
        Chem.SanitizeMol(molecule, sanitizeOps=Chem.SanitizeFlags.SANITIZE_ALL ^ flag)
    Chem.AssignStereochemistry(molecule, cleanIt=True, force=True)
    return molecule

def graph_from_molecule(molecule):
    """Convert RDKit molecule to PyTorch Geometric graph representation."""
    atom_features = []
    bond_features = []
    edge_index = []
    edge_attr = []

    # Chem.SanitizeMol(molecule)  # Ensure the molecule is sanitized

    for atom in molecule.GetAtoms():
        atom_features.append(atom_featurizer.encode(atom))  # Encode atom features

    for bond in molecule.GetBonds():
        start, end = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        edge_index.append([start, end])
        edge_index.append([end, start])  # Ensure undirected edges
        bond_features.append(bond_featurizer.encode(bond))  # Bond features
        bond_features.append(bond_featurizer.encode(bond))  # Reverse edge

    # Convert to PyTorch tensors
    x = torch.tensor(atom_features, dtype=torch.float)
    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(bond_features, dtype=torch.float)

    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr)

def graphs_from_smiles(smiles_list):
    """Convert a list of SMILES strings to PyTorch Geometric Data objects."""
    graphs = []
    for smiles in smiles_list:
        molecule = molecule_from_smiles(smiles)
        graph = graph_from_molecule(molecule)
        graphs.append(graph)
    return graphs  # This can be used with a PyG DataLoader


In [19]:
from pymatgen.core import Structure
from pymatgen.io.cif import CifParser
from pymatgen.analysis.local_env import CutOffDictNN
from scipy.spatial import distance_matrix
from rdkit import Chem
from rdkit.Chem import AllChem, Draw

import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

def cif_to_mol(cif_file):
    structure = Structure.from_file(cif_file)

    mol = Chem.RWMol()  # Create an editable molecule in RDKit

    for site in structure:
        element = site.specie.symbol  # Get atomic symbol (e.g., "C", "O")
        atom = Chem.Atom(element)  # Create an RDKit atom
        mol.AddAtom(atom)  # Add it to the molecule

    #print(structure[0].coords)

    cutoff = 3 # Example: typical bond length for C-C or C-H

    for i in range(len(structure)):
        for j in range(i + 1, len(structure)):
            dist = structure.get_distance(i, j)
            if dist < cutoff:  # Only consider distances within the cutoff
                #print(f"Bond between atom {i} and atom {j}: {dist:.3f} Å")
                if mol.GetBondBetweenAtoms(i, j) is None:  
                    mol.AddBond(i, j, Chem.BondType.SINGLE)  # Only add if not present

    conf = Chem.Conformer(mol.GetNumAtoms())  # Create a conformer

    for idx, site in enumerate(structure):
        coord = site.coords  # Cartesian coordinates (x, y, z)
        conf.SetAtomPosition(idx, coord)  # Assign position

    mol.AddConformer(conf)  # Add conformer to molecule

    return mol

In [20]:
from torch_geometric.data import Data

class PairedData(Data):
    def __init__(self, data1, data2, y):
        super().__init__()
        self.x1 = data1.x
        self.edge_index1 = data1.edge_index
        self.edge_attr1 = data1.edge_attr
        
        self.x2 = data2.x
        self.edge_index2 = data2.edge_index
        self.edge_attr2 = data2.edge_attr
        
        self.y = float(str(y).replace("−", "+"))  # Target value for the pair

    def __inc__(self, key, value, *args, **kwargs):
        """Ensures proper indexing when batching."""
        if key == "edge_index1":
            return self.x1.shape[0] if self.x1 is not None else 0
        if key == "edge_index2":
            return self.x2.shape[0] if self.x2 is not None else 0
        return super().__inc__(key, value, *args, **kwargs)
    


In [21]:
sdf_path = r"C:\Users\spran\Desktop\Mini-Projects\Drug Discovery Notebook\Basics\molecule.sdf"
cif_path = r"C:\Users\spran\Desktop\Mini-Projects\Drug Discovery Notebook\MM Interaction\CIF_files\BC3.cif"

supplier = Chem.SDMolSupplier(sdf_path)
mol2 = supplier[0]
graph2 = graph_from_molecule(mol2)

print(graph2)

mol = cif_to_mol(cif_path)
graph = graph_from_molecule(mol)

print(graph)

paired_Data = PairedData(graph, graph2, 3)

print(paired_Data)

Data(x=[11, 31], edge_index=[2, 22], edge_attr=[22, 7])
Data(x=[8, 31], edge_index=[2, 56], edge_attr=[56, 7])
PairedData(x1=[8, 31], edge_index1=[2, 56], edge_attr1=[56, 7], x2=[11, 31], edge_index2=[2, 22], edge_attr2=[22, 7], y=3.0)


In [22]:
import pandas as pd
import os

df = pd.read_csv(r"trial_dataCopy.csv") #TODO

file_path_cif = r"CIF_files" 
file_path_sdf = r"SDF_files" 

materials = []
drugs = []

for cif in df["material"]:
    file = os.path.join(file_path_cif, cif + ".cif")
    print(f"Processing CIF: {file}")  # Progress tracking
    mol = cif_to_mol(file)
    materials.append(graph_from_molecule(mol))
print("Finished processing all CIF files.\n")

for sdf in df["drug"]:
    file = os.path.join(file_path_sdf, sdf + ".sdf")
    print(f"Processing SDF: {file}")  # Progress tracking
    supplier = Chem.SDMolSupplier(file)
    mol = supplier[0]
    if mol is None:
        print(f"Warning: Failed to read molecule from {file}")  # Handle errors
    drugs.append(graph_from_molecule(mol))
print("Finished processing all SDF files.\n")

print("Pairing materials and drugs...")
paired_data_list = []
for i, (mat, drug, target) in enumerate(zip(materials, drugs, df["y"])):
    print(f"Pairing {i+1}/{len(df)}: Material-{i}, Drug-{i}, Target-{target}")
    print(f"Material graph: {mat}, Drug graph: {drug}")  # Debugging output
    paired_data_list.append(PairedData(mat, drug, target))
    #paired_data_list.append(PairedData(drug, mat, target))

print("Finished pairing all data.")


Processing CIF: CIF_files\BC3.cif
Processing CIF: CIF_files\BC3.cif
Processing CIF: CIF_files\Biphenylene.cif
Processing CIF: CIF_files\BN.cif
Processing CIF: CIF_files\BN.cif
Processing CIF: CIF_files\BN.cif
Processing CIF: CIF_files\BNNT.cif
Processing CIF: CIF_files\BNNT.cif
Processing CIF: CIF_files\graphene.cif
Processing CIF: CIF_files\graphene.cif
Processing CIF: CIF_files\graphene.cif
Processing CIF: CIF_files\Graphyne.cif
Processing CIF: CIF_files\Graphyne.cif
Processing CIF: CIF_files\Twin-Gr.cif
Processing CIF: CIF_files\Twin-Gr.cif
Processing CIF: CIF_files\Twin-Gr.cif
Processing CIF: CIF_files\Twin-Gr.cif
Processing CIF: CIF_files\molybdenum_disulfide.cif
Processing CIF: CIF_files\molybdenum_disulfide.cif
Processing CIF: CIF_files\phosphorene.cif
Processing CIF: CIF_files\phosphorene.cif
Processing CIF: CIF_files\phosphorene.cif
Processing CIF: CIF_files\phosphorene.cif
Processing CIF: CIF_files\phosphorene.cif
Processing CIF: CIF_files\phosphorene.cif
Processing CIF: CIF_

In [23]:
from sklearn.model_selection import train_test_split

# Define train-test split ratio (e.g., 80% train, 20% test)
train_data, test_data = train_test_split(paired_data_list, test_size=0.33, random_state=4)

print(f"Training set size: {len(train_data)}")
print(f"Testing set size: {len(test_data)}")

print(train_data)

Training set size: 24
Testing set size: 13
[PairedData(x1=[18, 31], edge_index1=[2, 210], edge_attr1=[210, 7], x2=[14, 31], edge_index2=[2, 28], edge_attr2=[28, 7], y=0.6), PairedData(x1=[8, 31], edge_index1=[2, 24], edge_attr1=[24, 7], x2=[15, 31], edge_index2=[2, 30], edge_attr2=[30, 7], y=0.63), PairedData(x1=[4, 31], edge_index1=[2, 8], edge_attr1=[8, 7], x2=[19, 31], edge_index2=[2, 40], edge_attr2=[40, 7], y=0.2), PairedData(x1=[4, 31], edge_index1=[2, 8], edge_attr1=[8, 7], x2=[15, 31], edge_index2=[2, 32], edge_attr2=[32, 7], y=1.397), PairedData(x1=[8, 31], edge_index1=[2, 56], edge_attr1=[56, 7], x2=[19, 31], edge_index2=[2, 38], edge_attr2=[38, 7], y=0.72), PairedData(x1=[2, 31], edge_index1=[2, 2], edge_attr1=[2, 7], x2=[18, 31], edge_index2=[2, 38], edge_attr2=[38, 7], y=0.15), PairedData(x1=[2, 31], edge_index1=[2, 2], edge_attr1=[2, 7], x2=[18, 31], edge_index2=[2, 38], edge_attr2=[38, 7], y=1.5), PairedData(x1=[80, 31], edge_index1=[2, 800], edge_attr1=[800, 7], x2=[9, 

In [24]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.nn import global_mean_pool as gap, global_max_pool as gmp

class MPNN(MessagePassing):
    def __init__(self, in_dim, edge_dim, out_dim, hidden_dim=32, aggr="max"):
        super().__init__(aggr=aggr)  # "mean", "sum", or "max" aggregation

        # MLP for message transformation
        self.mlp = nn.Sequential(
            nn.Linear(in_dim + edge_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, out_dim)
        )

    def forward(self, x, edge_index, edge_attr):
        # x: Node features (num_nodes, in_dim)
        # edge_index: Graph connectivity (2, num_edges)
        # edge_attr: Edge features (num_edges, edge_dim)

        return self.propagate(edge_index, x=x, edge_attr=edge_attr)

    def message(self, x_j, edge_attr):
        # x_j: Neighbor node features
        # edge_attr: Edge features

        # Concatenate node and edge features
        msg_input = torch.cat([x_j, edge_attr], dim=1)

        # Transform message using MLP
        return self.mlp(msg_input)

    def update(self, aggr_out):
        # Update node representations after aggregation
        return aggr_out


class CustomGNN(torch.nn.Module):
    def __init__(self, node_in_dim, edge_dim, hidden_dim, out_dim):
        super().__init__()
        self.conv1 = MPNN(node_in_dim, edge_dim, hidden_dim)
        self.conv2 = MPNN(hidden_dim, edge_dim, hidden_dim)
        self.conv3 = MPNN(hidden_dim, edge_dim, out_dim)

    def forward(self, x, edge_index, edge_attr):
        x = self.conv1(x, edge_index, edge_attr)
        x = F.relu(x)
        x = self.conv2(x, edge_index, edge_attr)
        x = F.relu(x)
        x = self.conv3(x, edge_index, edge_attr) 

        # Compute mean and max pooling
        hidden = torch.cat([gmp(x,batch=None), gap(x, batch=None)], dim=1)

        return hidden


In [25]:
class MLPMessagePassing(torch.nn.Module):
    def __init__(self, model,in_dim):
        super().__init__()
        self.GNN = model  # Pass an existing GNN model

        in_dim = 512

        self.MLP = nn.Sequential(
            nn.Linear(in_dim, in_dim // 2),  # First linear layer
            nn.ReLU(),  # Activation function
            nn.Linear(in_dim // 2, 1)  # Output layer
        )

    def forward(self, batch): #Equivariance 

        out1 = self.GNN(batch.x1, batch.edge_index1, batch.edge_attr1)
        out2 = self.GNN(batch.x2, batch.edge_index2, batch.edge_attr2)

        # Concatenate graph representations
        combined_out = torch.cat([out1, out2], dim=1)

        # Apply MLP for final output
        result = self.MLP(combined_out)

        return result

In [26]:
from sklearn.model_selection import train_test_split

# Define train-test split ratio (e.g., 80% train, 20% test)
train_data, test_data = train_test_split(paired_data_list, test_size=0.1, random_state=3)

print(f"Training set size: {len(train_data)}")
print(f"Testing set size: {len(test_data)}")

print(train_data[:5])

Training set size: 33
Testing set size: 4
[PairedData(x1=[3, 31], edge_index1=[2, 4], edge_attr1=[4, 7], x2=[10, 31], edge_index2=[2, 20], edge_attr2=[20, 7], y=0.74), PairedData(x1=[2, 31], edge_index1=[2, 2], edge_attr1=[2, 7], x2=[19, 31], edge_index2=[2, 38], edge_attr2=[38, 7], y=0.45), PairedData(x1=[18, 31], edge_index1=[2, 210], edge_attr1=[210, 7], x2=[14, 31], edge_index2=[2, 28], edge_attr2=[28, 7], y=0.6), PairedData(x1=[8, 31], edge_index1=[2, 24], edge_attr1=[24, 7], x2=[19, 31], edge_index2=[2, 38], edge_attr2=[38, 7], y=0.67), PairedData(x1=[18, 31], edge_index1=[2, 210], edge_attr1=[210, 7], x2=[9, 31], edge_index2=[2, 18], edge_attr2=[18, 7], y=0.43)]


In [ ]:
import torch.nn as nn
import torch.optim as optim
from torch_geometric.loader import DataLoader  # Assuming PyG DataLoader

# Define training function
def train(model, train_loader, optimizer, criterion, device):
    model.train()  # Set model to training mode
    total_loss = 0

    for batch in train_loader:

        batch = batch.to(device)  # Move batch to GPU if available
        
        optimizer.zero_grad()  # Reset gradients
        output = model(batch)  # Forward pass

        tensor_x = torch.tensor([batch.y], dtype=torch.float).to(device)  # Convert target to tensor 

        loss = torch.sqrt(criterion(output, tensor_x))  # RMSE Loss
        loss.backward()  # Backpropagation
        optimizer.step()  # Update weights

        total_loss += loss.item()
    
    return total_loss / len(train_loader)  # Return average loss

# Define evaluation function
def evaluate(model, val_loader, criterion, device):
    model.eval()  # Set model to evaluation mode
    total_loss = 0

    with torch.no_grad():  # Disable gradient tracking
        for batch in val_loader:
            batch = batch.to(device)
        
            output = model(batch)

            tensor_x = torch.tensor([batch.y], dtype=torch.float).to(device)  # Convert target to tensor 

            loss = torch.sqrt(criterion(output,tensor_x)) # RMSE Loss
            total_loss += np.absolute(loss.item())

    return total_loss / len(val_loader)

# Model, Optimizer, and Loss Function
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

gnn_model = CustomGNN(node_dim, edge_dim, hidden_dim=256, out_dim=128).to(device)
model = MLPMessagePassing(gnn_model, in_dim=256).to(device) #in_dim = out_dim x 2

optimizer = optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-3)
criterion = nn.MSELoss()  # RMSE = sqrt(MSE)

# Training Loop
num_epochs = 200
train_loader = train_data
val_loader = test_data

for data in train_data:
    print(data.edge_index1.shape, data.edge_index2.shape, data.y)  # Check the data structure

for epoch in range(1, num_epochs + 1):
    train_loss = train(model, train_loader, optimizer, criterion, device)
    val_loss = evaluate(model, val_loader, criterion, device)
    if (epoch%5 == 0): print(f"Epoch {epoch}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")

    #if (epoch%5 == 0): print(f"Epoch {epoch}: Train Loss = {train_loss:.4f}")

print("Training Complete! ✅")


In [ ]:
# # Training Loop
# num_epochs = 200
# train_loader = train_data
# val_loader = test_data

# for data in train_data:
#     print(data.edge_index1.shape, data.edge_index2.shape, data.y)  # Check the data structure

# for epoch in range(1, num_epochs + 1):
#     train_loss = train(model, train_loader, optimizer, criterion, device)
#     val_loss = evaluate(model, val_loader, criterion, device)
#     if (epoch%5 == 0): print(f"Epoch {epoch}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")

#     #if (epoch%5 == 0): print(f"Epoch {epoch}: Train Loss = {train_loss:.4f}")

# print("Training Complete! ✅")

In [36]:
from sklearn.model_selection import KFold
import torch
from torch.utils.data import DataLoader, Subset

k = 10 # Number of folds
num_epochs = 100  # Set as needed
kfold = KFold(n_splits=k, shuffle=True, random_state=69)

val_losses = []

for fold, (train_idx, val_idx) in enumerate(kfold.split(paired_data_list)):
    print(f"\nFold {fold+1}/{k}")

    # Split dataset
    train_loader = Subset(paired_data_list, train_idx)
    val_loader = Subset(paired_data_list, val_idx)

    print(f"Training set size: {len(train_loader)}")
    print(f"Validation set size: {len(val_loader)}")

    # Initialize model and optimizer **per fold**
    gnn_model = CustomGNN(node_dim, edge_dim, hidden_dim=256, out_dim=128).to(device)
    model = MLPMessagePassing(gnn_model, in_dim=256).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-5)
    criterion = torch.nn.MSELoss()  # or other criterion

    for epoch in range(0, num_epochs + 1):
        train_loss = train(model, train_loader, optimizer, criterion, device)
        
        if epoch % 10 == 0:
            val_loss = evaluate(model, val_loader, criterion, device)
            print(f"Epoch {epoch}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")
    
    val_losses.append(val_loss)

# Final result
mean_val_loss = sum(val_losses) / len(val_losses)
print(f"\n✅ K-Fold Cross-Validation Complete! Mean Validation Loss = {mean_val_loss:.4f}")


Fold 1/10
Training set size: 33
Validation set size: 4
Epoch 0: Train Loss = 0.5652, Val Loss = 0.3911
Epoch 10: Train Loss = 0.3531, Val Loss = 0.2324
Epoch 20: Train Loss = 0.3219, Val Loss = 0.2167
Epoch 30: Train Loss = 0.3281, Val Loss = 0.2591
Epoch 40: Train Loss = 0.3157, Val Loss = 0.2621
Epoch 50: Train Loss = 0.3118, Val Loss = 0.2506
Epoch 60: Train Loss = 0.3062, Val Loss = 0.2495
Epoch 70: Train Loss = 0.3104, Val Loss = 0.2689
Epoch 80: Train Loss = 0.3054, Val Loss = 0.2616
Epoch 90: Train Loss = 0.3080, Val Loss = 0.2572
Epoch 100: Train Loss = 0.3057, Val Loss = 0.2556

Fold 2/10
Training set size: 33
Validation set size: 4
Epoch 0: Train Loss = 0.4998, Val Loss = 0.3200
Epoch 10: Train Loss = 0.3099, Val Loss = 0.2032
Epoch 20: Train Loss = 0.2972, Val Loss = 0.2668
Epoch 30: Train Loss = 0.3177, Val Loss = 0.2052
Epoch 40: Train Loss = 0.3014, Val Loss = 0.2064
Epoch 50: Train Loss = 0.3162, Val Loss = 0.2240
Epoch 60: Train Loss = 0.3114, Val Loss = 0.2534
Epoch 7